In [11]:
# Phase 3: Machine Learning Modeling for Purchase Prediction
# Goal: Predict whether a user session will lead to a purchase (made_purchase = 1 or 0)

# ---------------------------
# Step 1: Load and prepare data
# ---------------------------

import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Load the session-level dataset from Phase 2
session_df = pd.read_csv(r"../phase2_OctNov_combined_session_features.csv")

print("Before Sampling:")
print(session_df['made_purchase'].value_counts(normalize=True))

# Sample size (recommended)
sample_size = 500_000

# Stratified sampling
if sample_size < len(session_df):
    session_df_sampled, _ = train_test_split(
        session_df,
        train_size=sample_size,
        stratify=session_df['made_purchase'],
        random_state=42
    )
else:
    session_df_sampled = session_df.copy()

session_df = session_df_sampled

print("\nAfter Sampling:")
print(session_df['made_purchase'].value_counts(normalize=True))

# ---------------------------
# Step 2: Define features and target
# ---------------------------

X = session_df.drop(columns=[
    'user_session',
    'user_id',
    'session_start',
    'session_end',
    'made_purchase'
])

y = session_df['made_purchase']

# ---------------------------
# Step 3: Train-test split
# ---------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ---------------------------
# Step 4: Random Forest doesn't require scaling
# ---------------------------

X_train_scaled = X_train
X_test_scaled = X_test

# Dummy scaler so dashboard loading doesn't break
scaler = None

# ---------------------------
# Step 5: Hyperparameter Tuning
# ---------------------------

param_dist = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15],
    'min_samples_split': [5, 10]
}

grid_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(
        class_weight='balanced',
        random_state=42
    ),
    param_distributions=param_dist,
    n_iter=4,
    cv=3,
    scoring='f1',
    verbose=2,
    n_jobs=2,
    random_state=42
)

grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_

print("\nBest Parameters:")
print(grid_search.best_params_)

# ---------------------------
# Step 6: Evaluate Model
# ---------------------------

y_pred = best_model.predict(X_test_scaled)

report = classification_report(y_test, y_pred)

print("\nClassification Report:")
print(report)

with open("classification_report.txt", "w") as f:
    f.write(report)

conf_matrix = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(conf_matrix)

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred))

plt.figure(figsize=(6,5))
sns.heatmap(conf_matrix,
            annot=True,
            fmt='d',
            cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("confusion_matrix.png")
plt.show()

# ---------------------------
# Step 7: Feature Importance
# ---------------------------

feature_importances = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop Feature Importances:")
print(feature_importances)

plt.figure(figsize=(10,6))
feature_importances.plot(kind='bar')

plt.title("Feature Importance")
plt.ylabel("Importance")

plt.tight_layout()
plt.savefig("feature_importance.png")
plt.show()

# ---------------------------
# Step 8: Save Model
# ---------------------------

joblib.dump(best_model, "purchase_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("\nModel and scaler saved successfully!")

MemoryError: Unable to allocate 256. KiB for an array with shape (32768,) and data type int64

In [ ]:
import pandas as pd

# Define 10 synthetic rows based on your testing
data = [
    [25, 10, 5, 1, 3, 2, 2000, 3000, 1500, 2, 600, 10],
    [5, 3, 0, 0, 1, 1, 500, 500, 500, 1, 120, 22],
    [50, 20, 10, 2, 5, 3, 1000, 1200, 800, 3, 900, 15],
    [8, 5, 0, 0, 2, 2, 100, 150, 50, 2, 300, 9],
    [15, 6, 2, 1, 4, 3, 2500, 4000, 1800, 4, 700, 14],
    [12, 4, 1, 0, 3, 2, 850, 950, 750, 2, 500, 18],
    [35, 18, 9, 1, 4, 3, 1800, 2500, 1200, 3, 800, 13],
    [4, 2, 0, 0, 1, 1, 300, 350, 250, 1, 100, 23],
    [20, 9, 4, 2, 3, 2, 1500, 1700, 1300, 2, 650, 11],
    [10, 6, 1, 0, 2, 2, 700, 800, 600, 2, 400, 16]
]

columns = [
    'num_events', 'num_views', 'num_carts', 'num_remove_from_cart',
    'num_unique_products', 'num_unique_categories',
    'avg_price', 'max_price', 'min_price',
    'num_brands', 'session_duration', 'hour_of_day'
]

df = pd.DataFrame(data, columns=columns)
excel_path = r"D:\Sparkathon\Data\synthetic_input.xlsx"
df.to_excel(excel_path, index=False)
print("Excel saved to:", excel_path)


Excel saved to: D:\Sparkathon\Data\synthetic_input.xlsx


In [ ]:
import pandas as pd
import joblib

# === Load model and scaler ===
model = joblib.load("purchase_model.pkl")
scaler = joblib.load("scaler.pkl")

# === Load the synthetic data from Excel ===
input_path = r"D:\Sparkathon\Data\synthetic_input.xlsx"
synthetic_data = pd.read_excel(input_path)

# === Scale the features ===
scaled_data = scaler.transform(synthetic_data)

# === Predict class and probabilities ===
predicted_classes = model.predict(scaled_data)
purchase_prob = model.predict_proba(scaled_data)[:, 1]

# === Combine results with input ===
results = synthetic_data.copy()
results["Predicted_Class"] = predicted_classes
results["Purchase_Probability"] = purchase_prob.round(4)

# === Save results to CSV ===
output_path = r"D:\Sparkathon\Data\synthetic_predictions_output.csv"
results.to_csv(output_path, index=False)
print("Results saved to:", output_path)

# Optional: Show preview
print("\nPrediction Results:\n")
print(results[["Predicted_Class", "Purchase_Probability"]])


Results saved to: D:\Sparkathon\Data\synthetic_predictions_output.csv

Prediction Results:

   Predicted_Class  Purchase_Probability
0                1                0.8502
1                1                0.9325
2                1                0.9339
3                1                0.8518
4                1                0.7318
5                1                0.9567
6                1                0.8557
7                1                0.9843
8                1                0.9576
9                1                0.9554
